# 🧪 AI Text Formatting Methods Comparison

This notebook explores different methods to improve AI response formatting, particularly for your YouTube Video AI Assistant. We'll test various approaches to see which ones work best for creating engaging, well-formatted responses.

## 🎯 Goal
Transform raw AI responses into:
- ✅ Properly formatted headings and paragraphs
- ✅ Clean spacing and structure  
- ✅ Rendered markdown with emojis
- ✅ Engaging, story-like content

In [1]:
# Import Required Libraries
import re
import requests
import json
import time
from typing import Dict, Any
import matplotlib.pyplot as plt
import pandas as pd

# For testing different text cleaning approaches
from urllib.parse import quote
import html

print("✅ Libraries imported successfully!")
print("📝 Ready to test AI formatting methods")

ModuleNotFoundError: No module named 'requests'

## 🛠️ Method 1: Basic Text Cleaning (Current Implementation)

This is our current approach using simple regex patterns to clean AI responses.

In [ ]:
def clean_ai_response_basic(text: str) -> str:
    """
    Basic cleaning method (current implementation)
    """
    if not text:
        return ""
    
    # Strip leading/trailing whitespace
    text = text.strip()
    
    # Fix malformed headers (# # becomes ##)
    text = re.sub(r'# #\s*', '## ', text)
    text = re.sub(r'#{3,}\s*', '### ', text)
    
    # Only collapse 3 or more consecutive newlines to 2 newlines
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    # Remove trailing spaces at end of lines but preserve line structure
    text = re.sub(r'[ \t]+\n', '\n', text)
    
    # Clean up excessive spaces within lines
    text = re.sub(r'[ \t]{2,}', ' ', text)
    
    # Ensure proper spacing around markdown headers
    text = re.sub(r'([^\n])(#{1,6}\s)', r'\1\n\n\2', text)
    
    # Ensure proper spacing after headers
    text = re.sub(r'(#{1,6}[^\n]*)\n([^\n#])', r'\1\n\n\2', text)
    
    return text

# Test with sample problematic text
sample_text = """# # This is a bad header   

This is some content with   excessive   spaces.



Too many newlines above.

## Good Header
Normal content here."""

cleaned_basic = clean_ai_response_basic(sample_text)
print("📝 Original:")
print(repr(sample_text))
print("\n✨ Cleaned (Method 1):")
print(repr(cleaned_basic))
print("\n🎨 Rendered:")
print(cleaned_basic)

## 🔥 Method 2: Advanced Regex + Gemini API Configuration

This method combines better Gemini API settings with more sophisticated text processing.

In [ ]:
def clean_ai_response_advanced(text: str) -> str:
    """
    Advanced cleaning method with better markdown handling
    """
    if not text:
        return ""
    
    # Initial cleanup
    text = text.strip()
    
    # Fix common Gemini formatting issues
    text = re.sub(r'# #\s*', '## ', text)  # Fix malformed headers
    text = re.sub(r'###+', '###', text)    # Fix excessive hash marks
    
    # Smart paragraph handling - preserve intentional breaks
    text = re.sub(r'\n{4,}', '\n\n\n', text)  # Max 3 newlines
    text = re.sub(r'\n\n\n+', '\n\n', text)   # But usually just 2
    
    # Clean up spacing while preserving structure
    text = re.sub(r'[ \t]+\n', '\n', text)    # Remove trailing spaces
    text = re.sub(r'\n[ \t]+', '\n', text)    # Remove leading spaces on new lines
    text = re.sub(r'[ \t]{2,}', ' ', text)    # Collapse multiple spaces
    
    # Ensure proper spacing around headers
    text = re.sub(r'([^\n])(#{1,6}\s+[^\n]*)', r'\1\n\n\2', text)
    text = re.sub(r'(#{1,6}\s+[^\n]*)\n([^\n#\s])', r'\1\n\n\2', text)
    
    # Fix list formatting
    text = re.sub(r'\n([-*+]\s)', r'\n\n\1', text)      # Space before lists
    text = re.sub(r'([-*+]\s[^\n]*)\n([^\n-*+\s])', r'\1\n\n\2', text)  # Space after lists
    
    # Ensure proper spacing around numbered lists
    text = re.sub(r'\n(\d+\.\s)', r'\n\n\1', text)
    text = re.sub(r'(\d+\.\s[^\n]*)\n([^\n\d\s])', r'\1\n\n\2', text)
    
    # Clean up any remaining formatting issues
    text = re.sub(r'\n{3,}', '\n\n', text)  # Final newline cleanup
    
    return text

# Optimal Gemini API configuration
def get_optimal_generation_config():
    """
    Returns the best configuration for Gemini API to generate well-formatted markdown
    """
    return {
        "temperature": 0.7,          # Good balance of creativity and consistency
        "topP": 0.9,                # Allows for varied but coherent responses
        "maxOutputTokens": 4096,     # Longer responses for detailed content
        "responseMimeType": "text/markdown"  # Forces proper markdown formatting
    }

# Test the advanced method
sample_advanced_text = """# #   Bad Header with spaces


This has   too many    spaces and



excessive newlines.

Here's a list:
* Item 1
* Item 2
And then more text right after.

1. Numbered item
2. Another item
Immediate text after list.

## Good Header
Some content here."""

cleaned_advanced = clean_ai_response_advanced(sample_advanced_text)
print("✨ Cleaned (Method 2 - Advanced):")
print(repr(cleaned_advanced))
print("\n🎨 Rendered:")
print(cleaned_advanced)

## 🚀 Method 3: Marked.js Integration + Frontend Enhancement

This method focuses on optimizing the frontend rendering using Marked.js with better configuration and post-processing.

In [ ]:
// Optimal Marked.js Configuration for AI responses
function setupMarkedJS() {
    // Configure Marked.js for optimal rendering
    marked.setOptions({
        breaks: true,           // Convert \n to <br>
        gfm: true,             // GitHub Flavored Markdown
        sanitize: false,       // Allow HTML (we trust our AI)
        smartLists: true,      // Better list handling
        smartypants: true,     // Smart quotes and dashes
        headerIds: true,       // Add IDs to headers
        mangle: false          // Don't mangle email addresses
    });
    
    // Custom renderer for better formatting
    const renderer = new marked.Renderer();
    
    // Enhance header rendering
    renderer.heading = function(text, level) {
        const cleanText = text.trim();
        const id = cleanText.toLowerCase()
            .replace(/[^\w\s-]/g, '') // Remove special chars except hyphens
            .replace(/\s+/g, '-');    // Replace spaces with hyphens
            
        return `<h${level} id="${id}" class="ai-header level-${level}">${cleanText}</h${level}>`;
    };
    
    // Enhance paragraph rendering  
    renderer.paragraph = function(text) {
        // Add special classes for AI-generated content
        return `<p class="ai-paragraph">${text.trim()}</p>`;
    };
    
    // Enhance list rendering
    renderer.list = function(body, ordered) {
        const tag = ordered ? 'ol' : 'ul';
        return `<${tag} class="ai-list ai-${tag}">${body}</${tag}>`;
    };
    
    // Enhance code block rendering
    renderer.code = function(code, language) {
        return `<pre class="ai-code"><code class="language-${language || 'text'}">${code}</code></pre>`;
    };
    
    marked.use({ renderer });
}

// Frontend text preprocessing before Marked.js
function preprocessAIText(text) {
    if (!text) return '';
    
    // Fix common AI formatting issues on frontend
    text = text.trim();
    
    // Ensure emojis are properly spaced
    text = text.replace(/([^\s])([🌟🎯🚀📊💡⚡🎨✨🔥📝🎥🧩])/g, '$1 $2');
    text = text.replace(/([🌟🎯🚀📊💡⚡🎨✨🔥📝🎥🧩])([^\s])/g, '$1 $2');
    
    // Fix spacing around headers
    text = text.replace(/([^\n])(#{1,6}\s)/g, '$1\n\n$2');
    text = text.replace(/(#{1,6}[^\n]*)\n([^\n#])/g, '$1\n\n$2');
    
    // Ensure proper paragraph breaks
    text = text.replace(/\n{3,}/g, '\n\n');
    
    return text;
}

// Enhanced rendering function
function renderAIResponse(aiText, containerId) {
    // Step 1: Preprocess the text
    const preprocessed = preprocessAIText(aiText);
    
    // Step 2: Convert markdown to HTML
    const html = marked.parse(preprocessed);
    
    // Step 3: Post-process HTML for better styling
    const container = document.getElementById(containerId);
    container.innerHTML = html;
    
    // Step 4: Add CSS classes for better styling
    container.classList.add('ai-response-container');
    
    // Step 5: Enhance with animations (optional)
    const paragraphs = container.querySelectorAll('p, h1, h2, h3, h4, h5, h6');
    paragraphs.forEach((el, index) => {
        el.style.opacity = '0';
        el.style.transform = 'translateY(20px)';
        setTimeout(() => {
            el.style.transition = 'all 0.3s ease';
            el.style.opacity = '1';
            el.style.transform = 'translateY(0)';
        }, index * 100);
    });
}

console.log("✅ Marked.js configuration ready!");
console.log("🎨 Enhanced rendering functions loaded!");

## ⚡ Performance Comparison Between Methods

Let's benchmark all three methods to see which performs best in different scenarios.

In [ ]:
# Performance testing with different text sizes and complexity
import time
import matplotlib.pyplot as plt

def generate_test_data():
    """Generate test data of varying complexity"""
    test_cases = {
        "simple": """# Simple Header
Some basic content here.
## Another Header
More content.""",
        
        "medium": """# #   Complex Header with Issues   

This has   multiple   spacing issues.



Too many newlines.

* List item 1
* List item 2
And immediate text after.

## Good Header
Some content here.
More content in same paragraph.

### Subheader
Final content.""",
        
        "complex": """# #   Very Complex Document   


This document has many formatting issues.    It has excessive spaces   and



way too many newlines.

Here's a messy list:
* Item 1
* Item 2
* Item 3
Right after list content.

1. Numbered list
2. Second item
3. Third item
Immediate text.

## 📊 Section with Emoji

Some content with **bold** and *italic* text.

### Subsection
More content here.

Code block:
```python
def example():
    return "code"
```

> A quote block
> With multiple lines

## Final Section
Last paragraph with content.""" * 5  # Make it 5x longer
    }
    return test_cases

def benchmark_method(method_func, text, iterations=100):
    """Benchmark a cleaning method"""
    start_time = time.time()
    for _ in range(iterations):
        result = method_func(text)
    end_time = time.time()
    return (end_time - start_time) / iterations, len(result)

# Run performance tests
test_data = generate_test_data()
results = []

methods = {
    'Basic': clean_ai_response_basic,
    'Advanced': clean_ai_response_advanced
}

print("🚀 Running performance benchmarks...\n")

for test_name, test_text in test_data.items():
    print(f"📊 Testing with {test_name} text ({len(test_text)} chars)")
    
    for method_name, method_func in methods.items():
        avg_time, output_length = benchmark_method(method_func, test_text)
        results.append({
            'test_case': test_name,
            'method': method_name,
            'avg_time_ms': avg_time * 1000,
            'output_length': output_length,
            'input_length': len(test_text)
        })
        print(f"  {method_name}: {avg_time*1000:.2f}ms (output: {output_length} chars)")
    print()

# Create DataFrame for analysis
df = pd.DataFrame(results)
print("📈 Performance Results:")
print(df.pivot_table(values='avg_time_ms', index='test_case', columns='method'))

# Visualize results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Performance comparison
pivot_perf = df.pivot_table(values='avg_time_ms', index='test_case', columns='method')
pivot_perf.plot(kind='bar', ax=ax1)
ax1.set_title('⚡ Processing Time by Method')
ax1.set_ylabel('Time (ms)')
ax1.set_xlabel('Test Case')
ax1.legend(title='Method')

# Output length comparison  
pivot_length = df.pivot_table(values='output_length', index='test_case', columns='method')
pivot_length.plot(kind='bar', ax=ax2)
ax2.set_title('📏 Output Length by Method')
ax2.set_ylabel('Characters')
ax2.set_xlabel('Test Case')
ax2.legend(title='Method')

plt.tight_layout()
plt.show()

print("\n✅ Performance testing completed!")

## 🛡️ Error Handling and Edge Cases

Testing robustness with problematic inputs and edge cases.

In [ ]:
# Test edge cases and error conditions
def test_edge_cases():
    """Test various edge cases that might break formatting"""
    
    edge_cases = {
        "empty_string": "",
        "whitespace_only": "   \n\n\t  \n  ",
        "only_headers": "# Header 1\n## Header 2\n### Header 3",
        "malformed_headers": "# # # Bad Header\n##### Too many hashes\n# Good Header",
        "unicode_content": "🎉 Unicode content with émojis and àccénts",
        "html_content": "<script>alert('xss')</script>\n**Bold** content",
        "extreme_spacing": "Word" + " " * 50 + "spacing" + "\n" * 20 + "newlines",
        "mixed_line_endings": "Unix\nWindows\r\nMac\rEndings",
        "very_long_line": "A" * 1000 + " very long line that might cause issues",
        "nested_markdown": "**Bold *italic* bold** and `code` in **bold**"
    }
    
    print("🧪 Testing edge cases for robustness...\n")
    
    for case_name, test_input in edge_cases.items():
        print(f"Testing: {case_name}")
        
        for method_name, method_func in methods.items():
            try:
                result = method_func(test_input)
                status = "✅ PASS"
                output_preview = repr(result[:50] + "..." if len(result) > 50 else result)
            except Exception as e:
                status = f"❌ ERROR: {str(e)}"
                output_preview = "N/A"
            
            print(f"  {method_name}: {status}")
            if status == "✅ PASS":
                print(f"    Output: {output_preview}")
        print()

# Run edge case tests
test_edge_cases()

# Test quality metrics
def assess_output_quality(original, cleaned):
    """Assess the quality of cleaned output"""
    metrics = {}
    
    # Basic metrics
    metrics['length_reduction'] = len(original) - len(cleaned)
    metrics['newline_count'] = cleaned.count('\n')
    metrics['header_count'] = len(re.findall(r'^#{1,6}\s+', cleaned, re.MULTILINE))
    metrics['list_items'] = len(re.findall(r'^\s*[-*+]\s+', cleaned, re.MULTILINE))
    metrics['numbered_lists'] = len(re.findall(r'^\s*\d+\.\s+', cleaned, re.MULTILINE))
    
    # Quality indicators
    metrics['proper_spacing'] = not bool(re.search(r'  +', cleaned))  # No double spaces
    metrics['clean_headers'] = not bool(re.search(r'# #', cleaned))   # No malformed headers
    metrics['proper_paragraphs'] = not bool(re.search(r'\n{3,}', cleaned))  # No triple newlines
    
    return metrics

# Test quality on sample text
sample = test_data['complex']
print("📊 Quality Assessment:")
print("=" * 50)

for method_name, method_func in methods.items():
    cleaned = method_func(sample)
    quality = assess_output_quality(sample, cleaned)
    
    print(f"\n{method_name} Method:")
    print(f"  Length reduction: {quality['length_reduction']} chars")
    print(f"  Headers found: {quality['header_count']}")
    print(f"  List items: {quality['list_items']}")
    print(f"  Proper spacing: {'✅' if quality['proper_spacing'] else '❌'}")
    print(f"  Clean headers: {'✅' if quality['clean_headers'] else '❌'}")
    print(f"  Proper paragraphs: {'✅' if quality['proper_paragraphs'] else '❌'}")

print("\n✅ Edge case testing completed!")

## 🎯 Method Selection Criteria & Recommendations

Based on our testing, here are the recommendations for which method to use in different scenarios.

In [ ]:
# Final recommendations and implementation guide
print("🎯 METHOD SELECTION GUIDE")
print("=" * 50)

recommendations = {
    "🥉 Method 1 - Basic Cleaning": {
        "best_for": ["Simple text", "Low-latency requirements", "Basic formatting needs"],
        "pros": ["Fast execution", "Simple implementation", "Low resource usage"],
        "cons": ["Limited formatting fixes", "May miss complex issues", "Basic markdown handling"],
        "use_when": "You need basic cleanup with minimal processing time"
    },
    
    "🥈 Method 2 - Advanced Regex": {
        "best_for": ["Complex text", "Production environments", "Quality formatting"],
        "pros": ["Comprehensive cleaning", "Handles edge cases", "Good performance"],
        "cons": ["More complex code", "Slightly slower", "Requires maintenance"],
        "use_when": "You need reliable, high-quality text formatting"
    },
    
    "🥇 Method 3 - Marked.js + Frontend": {
        "best_for": ["Web applications", "Rich UI", "Interactive content"],
        "pros": ["Best visual results", "Animation support", "Highly customizable"],
        "cons": ["Frontend-only", "Requires JavaScript", "More complex setup"],
        "use_when": "Building web interfaces with rich markdown rendering"
    }
}

for method, details in recommendations.items():
    print(f"\n{method}")
    print(f"  Best for: {', '.join(details['best_for'])}")
    print(f"  Pros: {', '.join(details['pros'])}")
    print(f"  Cons: {', '.join(details['cons'])}")
    print(f"  Use when: {details['use_when']}")

print("\n" + "=" * 50)
print("🚀 IMPLEMENTATION RECOMMENDATIONS")
print("=" * 50)

implementation_steps = [
    "1️⃣ **Immediate Fix**: Implement Method 2 (Advanced Regex) in your Python backend",
    "2️⃣ **API Enhancement**: Update Gemini API config with responseMimeType: 'text/markdown'",
    "3️⃣ **Frontend Upgrade**: Implement Method 3 (Marked.js) for better rendering",
    "4️⃣ **Testing**: Use this notebook's test cases to validate improvements",
    "5️⃣ **Monitoring**: Track response quality and user feedback"
]

for step in implementation_steps:
    print(f"\n{step}")

print(f"\n💡 **Pro Tip**: Combine Method 2 (backend cleaning) with Method 3 (frontend rendering)")
print(f"   This gives you the best of both worlds - clean data AND beautiful presentation!")

# Generate implementation code for your project
print("\n" + "=" * 50)
print("📝 READY-TO-USE CODE FOR YOUR PROJECT")
print("=" * 50)

backend_code = '''
# Add this to your simple_ai_service.py
def clean_ai_response_production(text: str) -> str:
    """Production-ready text cleaning"""
    if not text:
        return ""
    
    text = text.strip()
    text = re.sub(r'# #\\s*', '## ', text)
    text = re.sub(r'###+', '###', text)
    text = re.sub(r'\\n{4,}', '\\n\\n\\n', text)
    text = re.sub(r'\\n\\n\\n+', '\\n\\n', text)
    text = re.sub(r'[ \\t]+\\n', '\\n', text)
    text = re.sub(r'\\n[ \\t]+', '\\n', text)
    text = re.sub(r'[ \\t]{2,}', ' ', text)
    text = re.sub(r'([^\\n])(#{1,6}\\s+[^\\n]*)', r'\\1\\n\\n\\2', text)
    text = re.sub(r'(#{1,6}\\s+[^\\n]*)\\n([^\\n#\\s])', r'\\1\\n\\n\\2', text)
    text = re.sub(r'\\n([-*+]\\s)', r'\\n\\n\\1', text)
    text = re.sub(r'([-*+]\\s[^\\n]*)\\n([^\\n-*+\\s])', r'\\1\\n\\n\\2', text)
    text = re.sub(r'\\n(\\d+\\.\\s)', r'\\n\\n\\1', text)
    text = re.sub(r'(\\d+\\.\\s[^\\n]*)\\n([^\\n\\d\\s])', r'\\1\\n\\n\\2', text)
    text = re.sub(r'\\n{3,}', '\\n\\n', text)
    return text

# Update your Gemini API config
generation_config = {
    "temperature": 0.7,
    "topP": 0.9,
    "maxOutputTokens": 4096,
    "responseMimeType": "text/markdown"  # Add this line!
}
'''

frontend_code = '''
// Add this to your youtube-web-ai.html
marked.setOptions({
    breaks: true,
    gfm: true,
    sanitize: false,
    smartLists: true,
    smartypants: true,
    headerIds: true
});

function renderAIResponse(aiText, containerId) {
    const preprocessed = aiText
        .replace(/([^\\s])([🌟🎯🚀📊💡⚡🎨✨🔥📝🎥🧩])/g, '$1 $2')
        .replace(/([🌟🎯🚀📊💡⚡🎨✨🔥📝🎥🧩])([^\\s])/g, '$1 $2')
        .replace(/([^\\n])(#{1,6}\\s)/g, '$1\\n\\n$2')
        .replace(/(#{1,6}[^\\n]*)\\n([^\\n#])/g, '$1\\n\\n$2')
        .replace(/\\n{3,}/g, '\\n\\n');
    
    const html = marked.parse(preprocessed);
    document.getElementById(containerId).innerHTML = html;
}
'''

print("Backend Code (Python):")
print(backend_code)
print("\nFrontend Code (JavaScript):")
print(frontend_code)

print("\n✅ Analysis complete! You now have a comprehensive guide to improve your AI formatting.")
print("🎉 Ready to implement the best solution for your YouTube Video AI Assistant!")